# Employee dataset analysis with PySpark

This notebook uses Apache Spark to import employee records, improve their quality, transform selected fields, and produce analytical summaries.

## Part 1: Configure the workspace and Spark

The required local settings are prepared before a Spark session is created.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Set the local Hadoop-related environment values
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Employee_Data_Processing") \
    .getOrCreate()

print("Spark Session initiated successfully!")
print("Spark Version:", spark.version)


## Part 2: Read and verify the input

The CSV is loaded, then its row count and inferred schema are checked.

In [ ]:
# Read the input CSV using the notebook's relative location
input_file = "../../Employee.csv"

employee_raw_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(input_file)

print(f"Ingested {employee_raw_df.count()} records.")
employee_raw_df.printSchema()
employee_raw_df.show(5, truncate=False)


## Part 3: Remove duplicates and complete missing values

Repeated rows are removed and selected null values receive defined replacement values.

In [ ]:
# Keep one copy of each repeated record
employee_clean_df = employee_raw_df.dropDuplicates()
print(f"Row count after deduplication: {employee_clean_df.count()}")

# Fill null fields with the selected defaults
fill_defaults = {
    "Education": "Undergrad",
    "City": "Unknown",
    "Gender": "Unknown",
    "PaymentTier": 3,
    "Age": 0,
    "ExperienceInCurrentDomain": 0
}
employee_clean_df = employee_clean_df.na.fill(fill_defaults)

# Confirm that no nulls remain in the target fields
employee_clean_df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in employee_clean_df.columns]
).show()


## Part 4: Set data types and apply selection rules

Fields are cast to the required types and the analysis keeps employees aged 25 or above.

In [ ]:
# Convert analysis fields to their intended types
employee_transformed_df = employee_clean_df \
    .withColumn("Age", F.col("Age").cast("integer")) \
    .withColumn("PaymentTier", F.col("PaymentTier").cast("double")) \
    .withColumn("ExperienceInCurrentDomain", F.col("ExperienceInCurrentDomain").cast("integer"))

# Retain rows that meet the analysis criteria
employee_filtered_df = employee_transformed_df \
    .filter((F.col("Age") >= 25) & F.col("City").isNotNull())

print(f"Records kept after filtering: {employee_filtered_df.count()}")


## Part 5: Produce descriptive measures

Aggregate measures are calculated for the retained employee records.

In [ ]:
# Calculate the selected summary statistics
employee_filtered_df.select(
    F.count("*").alias("Active_Employees"),
    F.round(F.avg("Age"), 2).alias("Average_Age"),
    F.min("Age").alias("Minimum_Age"),
    F.max("Age").alias("Maximum_Age"),
    F.round(F.avg("ExperienceInCurrentDomain"), 2).alias("Avg_Domain_Experience")
).show(truncate=False)


## Part 6: Compare employee groups

Education and gender groups are compared by headcount and average domain experience.

In [ ]:
# Compute metrics for each employee group
education_summary_df = employee_filtered_df \
    .groupBy("Education", "Gender") \
    .agg(
        F.count("*").alias("Staff_Count"),
        F.round(F.avg("ExperienceInCurrentDomain"), 2).alias("Average_Domain_Exp"),
        F.round(F.avg("PaymentTier"), 2).alias("Average_Payment_Tier")
    ) \
    .orderBy(F.desc("Staff_Count"))

education_summary_df.show(50, truncate=False)

# Filter the grouped results using the requested rule
high_experience_segments = education_summary_df.filter(F.col("Average_Domain_Exp") > 2.0)
high_experience_segments.show(truncate=False)


## Part 7: Save and validate the outputs

The cleaned and aggregated datasets are written to disk, then a quick read check confirms the output.

In [ ]:
# Set the destinations for generated files
output_clean = "../output/cleaned_data"
output_summary = "../output/grouped_results"

# Save the processed datasets
employee_clean_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_clean)

education_summary_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_summary)

print("Pipelines exported successfully!")

# Read the exported data to verify it
verification_df = spark.read.option("header", True).csv(output_clean)
verification_df.show(5)

spark.stop()
